In [ ]:
import os
import pandas as pd
from pysus import CNES

# Configurações
# DICA: Use apenas UM mês para representar o ano (Ex: Dezembro) para evitar duplicação.
ANOS = [2024]
MES_REFERENCIA = 12 
ESTADOS = ['SP', 'RJ']
caminho_base = './data/raw/cnes'

def processar_cnes():
    cnes = CNES().load()
    
    for uf in ESTADOS:
        for ano in ANOS:
            print(f'--- Processando {uf} - {ano}/{MES_REFERENCIA} ---')
            
            # ---------------------------------------------------------
            # PARTE 1: LEITOS (LT) - Mais leve
            # ---------------------------------------------------------
            try:
                # Busca arquivos do grupo 'LT' (Leitos)
                files_lt = cnes.get_files(group='LT', uf=uf, year=ano, month=MES_REFERENCIA)
                
                if files_lt:
                    path_lt = f"{caminho_base}/leitos"
                    os.makedirs(path_lt, exist_ok=True)
                    
                    parquets_lt = cnes.download(files_lt, local_dir=path_lt)
                    
                    for p in parquets_lt:
                        # Colunas: CNES (Estabelecimento), Município, Qtd Total, Qtd SUS
                        cols_lt = ['CNES', 'CO_MUNICIPIO', 'QT_EXIST', 'QT_SUS']
                        df_lt = pd.read_parquet(p, columns=cols_lt)
                        
                        # Agrupa por Município (Soma os leitos de todos hospitais da cidade)
                        df_lt_agrupado = df_lt.groupby('CO_MUNICIPIO')[['QT_EXIST', 'QT_SUS']].sum().reset_index()
                        
                        print(f"[{uf}] Leitos processados. Salvando resumo...")
                        df_lt_agrupado.to_csv(f"{path_lt}/resumo_leitos_{uf}_{ano}_{MES_REFERENCIA}.csv", index=False)
            except Exception as e:
                print(f"Erro em Leitos ({uf}): {e}")

            # ---------------------------------------------------------
            # PARTE 2: PROFISSIONAIS (PF) - Pesado! Cuidado.
            # ---------------------------------------------------------
            try:
                # Busca arquivos do grupo 'PF' (Profissionais)
                files_pf = cnes.get_files(group='PF', uf=uf, year=ano, month=MES_REFERENCIA)
                
                if files_pf:
                    path_pf = f"{caminho_base}/profissionais"
                    os.makedirs(path_pf, exist_ok=True)
                    
                    parquets_pf = cnes.download(files_pf, local_dir=path_pf)
                    
                    for p in parquets_pf:
                        print(f"[{uf}] Lendo Profissionais (isso pode demorar)...")
                        
                        # Lê apenas colunas essenciais
                        # CBO = Código Brasileiro de Ocupação (Médico começa com '225')
                        cols_pf = ['CNES', 'CO_MUNICIPIO', 'CBO'] 
                        df_pf = pd.read_parquet(p, columns=cols_pf)
                        
                        # FILTRO DE MÉDICOS: CBO começa com '225'
                        # Convertemos para string para garantir o startswith
                        df_medicos = df_pf[df_pf['CBO'].astype(str).str.startswith('225')]
                        
                        # Contagem: Quantos vínculos médicos existem por município
                        # Nota: Um médico pode trabalhar em 2 lugares, aqui contamos "força de trabalho" (vínculos)
                        df_medicos_agrupado = df_medicos.groupby('CO_MUNICIPIO').size().reset_index(name='TOTAL_MEDICOS')
                        
                        print(f"[{uf}] Médicos filtrados: {len(df_medicos)} vínculos encontrados.")
                        df_medicos_agrupado.to_csv(f"{path_pf}/resumo_medicos_{uf}_{ano}_{MES_REFERENCIA}.csv", index=False)
                        
            except Exception as e:
                print(f"Erro em Profissionais ({uf}): {e}")

if __name__ == "__main__":
    processar_cnes()
    print("Extração CNES concluída.")